In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

from sentence_transformers import SentenceTransformer

ROOT = Path.cwd().parent
print(ROOT)

DATA_RAW = ROOT/"data/processed"

INPUT_PATH = DATA_RAW / "the_numbers_plot_franchise.csv"
OUTPUT_PATH = DATA_RAW / "00_descr_title_df.csv"

model_df = scrape_df = pd.read_csv(INPUT_PATH)

/workspaces/BoxOffice-Oracle/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/workspaces/BoxOffice-Oracle


In [12]:
model_df["plot_summary"].notna().mean()

np.float64(0.7085492227979274)

In [29]:
sample_df=model_df[model_df["plot_summary"].notna()]

In [16]:
import re

def remove_title_from_plot(row):
    title = str(row["primaryTitle"])
    plot = str(row["plot_summary"])

    if not title or not plot:
        return plot

    pattern = re.escape(title)
    cleaned = re.sub(pattern, "", plot, flags=re.IGNORECASE)

    return re.sub(r"\s+", " ", cleaned).strip()

sample_df["plot_summary_no_title"] = sample_df.apply(
    remove_title_from_plot,
    axis=1
)

In [19]:
# Load small embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Fill missing text with empty string
plot_texts = sample_df["plot_summary_no_title"].fillna("").astype(str).tolist()

# Create embeddings
plot_embeddings = embedder.encode(
    plot_texts,
    batch_size=32,
    show_progress_bar=False, #Need to download extra stuff for it
    normalize_embeddings=True
)

plot_embeddings.shape

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10568.36it/s]


(1641, 384)

In [20]:
# Fill missing text with empty string
title_texts = sample_df["primaryTitle"].fillna("").astype(str).tolist()

# Create embeddings
title_embeddings = embedder.encode(
    title_texts,
    batch_size=32,
    show_progress_bar=False, #Need to download extra stuff for it
    normalize_embeddings=True
)

title_embeddings.shape

(1641, 384)

In [22]:
sample_df["title_plot_similarity"] = np.sum(
    title_embeddings * plot_embeddings,
    axis=1
)

In [23]:
sample_df.head()

,tconst,primaryTitle,startYear,the_numbers_url,scrape_success,scrape_error,plot_summary,franchise,has_plot_summary,title_plot_similarity,plot_summary_no_title
0,tt0120667,Fantastic Four,2005.0,https://www.the-numbers.com/movie/Fantastic-Fo...,True,NaN,Scientist Reed Richards persuades his arrogant...,Fantastic Four,True,0.217350,Scientist Reed Richards persuades his arrogant...
2,tt0121766,Star Wars: Episode III - Revenge of the Sith,2005.0,https://www.the-numbers.com/movie/Star-Wars-Ep...,True,NaN,"Years after the onset of the Clone Wars, the n...",Star Wars,True,0.473704,"Years after the onset of the Clone Wars, the n..."
4,tt0200465,The Bank Job,2008.0,https://www.the-numbers.com/movie/Bank-Job-The,True,NaN,Self-reformed petty criminal Terry Leather has...,NaN,True,0.257784,Self-reformed petty criminal Terry Leather has...
5,tt0204313,Exorcist: The Beginning,2004.0,https://www.the-numbers.com/movie/Exorcist-The...,True,NaN,Father Lankester Merrin lost his faith after t...,Exorcist,True,0.143046,Father Lankester Merrin lost his faith after t...
10,tt0257516,Cursed,2005.0,https://www.the-numbers.com/movie/Cursed,True,NaN,"In Los Angeles, siblings Ellie and Jimmy come ...",NaN,True,0.248333,"In Los Angeles, siblings Ellie and Jimmy come ..."


In [24]:
sample_df[
    [
        "primaryTitle",
        "plot_summary",
        "title_plot_similarity"
    ]
].sort_values(
    "title_plot_similarity",
    ascending=False
).head(20)

,primaryTitle,plot_summary,title_plot_similarity
1205,Mr. Popper's Penguins,"In this family comedy, Mr. Popper, is a driven...",0.833349
806,The Adventures of Tintin,Tintin is a young reporter whose relentless pu...,0.804434
1231,Hansel & Gretel: Witch Hunters,"After getting a taste for blood as children, H...",0.786131
1939,Godzilla: King of the Monsters,An action adventure that pits Godzilla against...,0.777971
1042,Rambo: Last Blood,"Almost four decades after he drew first blood,...",0.771783
2275,Kraven the Hunter,Kraven is a man whose complex relationship wit...,0.755683
1471,Horizon: An American Saga - Chapter 1,Horizon: An American Saga explores the lure of...,0.749925
1254,Godzilla X Kong: The New Empire,This latest entry follows up the explosive sho...,0.749787
146,"The Chronicles of Narnia: The Lion, the Witch ...",Four young adventurers playing hide and-seek i...,0.746720
2301,Shang-Chi and the Legend of the Ten Rings,Shang-Chi must confront the past he thought he...,0.744656


In [30]:
sample_df["franchise"].notna().mean()

np.float64(0.3583180987202925)

In [31]:
full_df = pd.read_csv(
    DATA_RAW/"the_numbers_model_base_v1.csv"
)

In [37]:
descr_title_df = full_df.merge(sample_df[["tconst","plot_summary","title_plot_similarity","franchise"]],how='inner',on="tconst")

In [39]:
descr_title_df["title_word_count"] = (
    descr_title_df["primaryTitle"]
    .fillna("")
    .str.split()
    .str.len()
)

descr_title_df["title_char_count"] = (
    descr_title_df["primaryTitle"]
    .fillna("")
    .str.len()
)

In [40]:
descr_title_df.sort_values(
    "title_plot_similarity",
    ascending=False
).head(20)

,tconst,primaryTitle,startYear,the_numbers_url,scrape_success,scrape_error,opening_weekend_gross,opening_theaters,domestic_release_date,release_type,...,legs,plot_point,raw_opening_weekend_text,raw_theater_counts_text,raw_domestic_releases_text,plot_summary,title_plot_similarity,franchise,title_word_count,title_char_count
570,tt1396218,Mr. Popper's Penguins,2011.0,https://www.the-numbers.com/movie/Mr-Poppers-P...,True,NaN,18445355.0,3339.0,2011-06-17,Wide,...,3.70,"Dysfunctional Family, Inheritance, Mid-Life Cr...","$18,445,355 (27.0% of total gross)","3,339 opening theaters/3,342 max. theaters, 4....","June 17th, 2011 (Wide) by 20th Century Fox, re...","In this family comedy, Mr. Popper, is a driven...",0.840938,NaN,3,21
728,tt1611224,Abraham Lincoln: Vampire Hunter,2012.0,https://www.the-numbers.com/movie/Abraham-Linc...,True,NaN,16306974.0,3108.0,2012-06-22,Wide,...,2.30,"Alternate History, Vampire, Young Child Dealin...","$16,306,974 (43.5% of total gross)","3,108 opening theaters/3,109 max. theaters, 3....","June 22nd, 2012 (Wide) by 20th Century Fox",Abraham Lincoln: Vampire Hunter brings to the ...,0.824134,NaN,4,31
1371,tt5090568,Transformers: Rise of the Beasts,2023.0,https://www.the-numbers.com/movie/Transformers...,True,NaN,61045464.0,3678.0,2023-06-09,Wide,...,2.58,Robot,"$61,045,464 (38.8% of total gross)","3,678 opening theaters/3,680 max. theaters, 5....","June 9th, 2023 (Wide) by Paramount Pictures Ju...",Returning to the action and spectacle that hav...,0.820709,Transformers,5,32
719,tt1598822,New Year's Eve,2011.0,https://www.the-numbers.com/movie/New-Years-Eve,True,NaN,13019180.0,3505.0,2011-12-09,Wide,...,4.19,"Coming of Age, Romance","$13,019,180 (23.9% of total gross)","3,505 opening theaters/3,505 max. theaters, 4....","December 9th, 2011 (Wide) by Warner Bros.","""New Year's Eve"" celebrates love, hope, forgiv...",0.813127,Garry Marshall's Holiday Franchise,3,14
153,tt0458339,Captain America: The First Avenger,2011.0,https://www.the-numbers.com/movie/Captain-Amer...,True,NaN,65058524.0,3715.0,2011-07-22,Wide,...,2.72,"Good vs. Evil, Nazis, Non-Chronological, Occul...","$65,058,524 (36.8% of total gross)","3,715 opening theaters/3,715 max. theaters, 5....","July 22nd, 2011 (Wide) by Paramount Pictures",Captain America: The First Avenger will focus ...,0.805694,Captain America,5,34
288,tt0983193,The Adventures of Tintin,2011.0,https://www.the-numbers.com/movie/Adventures-o...,True,NaN,9720993.0,3087.0,2011-12-21,Wide,...,6.78,"Addiction, Pirates, Treasure Hunters, Twins","$9,720,993 (12.5% of total gross)","3,087 opening theaters/3,087 max. theaters, 4....","December 21st, 2011 (Wide) by Paramount Pictur...",Tintin is a young reporter whose relentless pu...,0.804434,Tintin,4,24
794,tt1702443,Justin Bieber: Never Say Never,2011.0,https://www.the-numbers.com/movie/Justin-Biebe...,True,NaN,29514054.0,3105.0,2011-02-09,Wide,...,2.47,"Musicians, Non-Chronological","$29,514,054 (40.4% of total gross)","3,105 opening theaters/3,118 max. theaters, 4....","February 9th, 2011 (Special Engagement) by Par...",Justin Bieber: Never Say Never is the inspirin...,0.803139,NaN,5,30
242,tt0829150,Dracula Untold,2014.0,https://www.the-numbers.com/movie/Dracula-Untold,True,NaN,23514615.0,2887.0,2014-10-10,Wide,...,2.38,"Faustian, Monster, Origin Story, Vampire","$23,514,615 (42.0% of total gross)","2,887 opening theaters/2,900 max. theaters, 4....","October 10th, 2014 (Wide) by Universal October...",Dracula Untold is the origin story of the man ...,0.793902,Dark Universe,2,14
593,tt1428538,Hansel & Gretel: Witch Hunters,2013.0,https://www.the-numbers.com/movie/Hansel-and-G...,True,NaN,19690956.0,3372.0,2013-01-25,Wide,...,2.83,"Bounty Hunter, Corrupt Cops, Non-Chronological...","$19,690,956 (35.3% of total gross)","3,372 opening theaters/3,375 max. theaters, 4....","January 25th, 2013 (Wide) by Paramount Picture...","After getting a taste for blood as children, H...",0.786131,NaN,5,30
14,tt0320661,Kingdom of Heaven,2005.0,htt

In [41]:
descr_title_df.to_csv(
    OUTPUT_PATH,
    index=False
)